# Beyond CSV and Excel

**Author:** Margaret Gonzalez
**Course:** Data Science
**Date:** September 2026

Three data sources are used in this notebook:

1. `harry_potter.docx`, a Word document containing the full text of *Harry Potter and the Sorcerer's Stone*, sourced from [Kaggle](https://www.kaggle.com/). Loaded with a **relative path** so the notebook runs from its own folder.
2. The Wikipedia article [Illuminati (comics)](https://en.wikipedia.org/wiki/Illuminati_(comics)), parsed as HTML.
3. The free [Disney API](https://api.disneyapi.dev/character), which needs no credentials.

Run the cells in order. Parts 2 and 3 require an internet connection.

**Submission note:** the final item writes `disney_characters.xlsx` to this folder. That workbook is a
separate required upload alongside the zipped notebook.

## Setup

In [ ]:
# Install anything that is missing. Safe to re-run.
%pip install --quiet textract python-docx nltk wordcloud beautifulsoup4 lxml requests openpyxl

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import re
import json
import string
from collections import Counter
from urllib.parse import urljoin

import requests
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from lxml import html as lh
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords

# Download the stopword list once. Quiet so it does not spam the output.
nltk.download("stopwords", quiet=True)

plt.rcParams["figure.dpi"] = 110
print("Working directory:", os.getcwd())
print("pandas", pd.__version__)

---
# Part 1: Harry Potter Word document

### 1. Extract the text and display the first 350 characters (3 points)

The extraction returns a **bytes** object, so it has to be decoded into a regular character string
before it can be analyzed. `textract.process()` is the primary path. If `textract` will not install
on a given machine, the fallback uses `python-docx` and encodes the result so the rest of the
notebook receives an identical bytes object.

In [ ]:
DOCX_PATH = "harry_potter.docx"   # relative path, the file sits beside this notebook

try:
    import textract
    raw_bytes = textract.process(DOCX_PATH)
    source_used = "textract"
except Exception as exc:
    import docx
    document = docx.Document(DOCX_PATH)
    raw_bytes = "\n\n".join(p.text for p in document.paragraphs).encode("utf-8")
    source_used = f"python-docx fallback ({type(exc).__name__})"

print("Extractor used:", source_used)
print("Type before decoding:", type(raw_bytes))
print("Bytes extracted:", len(raw_bytes))

In [ ]:
# Still a byte string at this point. Notice the b'' prefix and the escape sequences.
raw_bytes[:350]

In [ ]:
# Convert from bytes to a character string
hp_text = raw_bytes.decode("utf-8")

print("Type after decoding:", type(hp_text))
print("Characters:", len(hp_text))

hp_text[:350]

### 2. Display the first 350 characters again using `print()` (3 points)

In [ ]:
print(hp_text[:350])

**What is different about the output?**

The plain expression in the previous cell returns the *representation* of the string, which is what
Jupyter echoes for the last line of a cell. It arrives wrapped in single quotes, squeezed onto one
long wrapped line, with every line break shown literally as the two characters `\n` and every
non-ASCII character shown as an escape code. In the byte version, the curly apostrophe in "you'd"
even appears as the three byte sequence `\xe2\x80\x99`.

`print()` sends the string to standard output instead, so Python interprets those escape sequences
rather than displaying them. The `\n` characters become real line breaks, the paragraph structure of
the novel becomes visible, the surrounding quotes disappear, and the curly apostrophe renders as the
character it actually is. The underlying string is identical in both cells. Only the display changes.

### 3. Remove punctuation and numbers, then display the first 300 characters (3 points)

In [ ]:
# string.punctuation covers the standard ASCII punctuation set
print("Punctuation being removed:", string.punctuation)

no_punct = "".join(ch for ch in hp_text if ch not in string.punctuation)
no_digits = "".join(ch for ch in no_punct if not ch.isdigit())

print()
print(no_digits[:300])

Two curly characters survive this step, the right single quotation mark in words such as `didn't`
and the right double quotation mark used for dialogue. They are Unicode punctuation and are not part
of `string.punctuation`, which only holds the 32 ASCII symbols. This is exactly the wrinkle the
assignment warns about in item 7.

### 4. Lowercase the text, split into words, display the first 25 (3 points)

In [ ]:
hp_words = no_digits.lower().split()

print("Total words:", len(hp_words))
hp_words[:25]

### 5. Remove English stopwords and words of three characters or less (4 points)

In [ ]:
stop_en = stopwords.words("english")
print("Stopwords in the NLTK English list:", len(stop_en))

stop_set = set(stop_en)
hp_filtered = [w for w in hp_words if w not in stop_set and len(w) > 3]

print("Words before filtering:", len(hp_words))
print("Words after filtering: ", len(hp_filtered))
print("Removed:", len(hp_words) - len(hp_filtered),
      f"({(len(hp_words) - len(hp_filtered)) / len(hp_words) * 100:.1f}%)")
print()
print(hp_filtered[:25])

In [ ]:
# Which of the first 25 original words disappeared, and why
dropped = [w for w in hp_words[:25] if w in stop_set or len(w) <= 3]
print("Dropped from the first 25 words:", dropped)
print()
print("Dropped because they are stopwords AND longer than three characters:",
      [w for w in dropped if w in stop_set and len(w) > 3])

**A couple of stopwords longer than three characters that were removed**

Comparing the two lists of 25, the opening line of the novel loses `were`, `that`, `they` and `very`.
All four are longer than three characters, so the length rule did not catch them. They disappeared
purely because they sit in the NLTK English stopword list. Short words such as `the`, `who`, `and`,
`of` and `to` were removed as well, but those would have been dropped by the three character rule
anyway, so they do not answer the question. `were`, `that`, `they` and `very` are the clean examples.

### 6. Display the top 30 unigrams with their counts (3 points)

In [ ]:
hp_counts = Counter(hp_filtered)
top30 = hp_counts.most_common(30)

pd.DataFrame(top30, columns=["Unigram", "Count"]).set_index("Unigram")

### 7. Remove `didn't`, `rowling` and `like`, then re-create the top 30 (4 points)

The apostrophe inside `didn't` is U+2019, the right single quotation mark, not the ASCII apostrophe.
It is copied directly from the output above so the match succeeds.

In [ ]:
remove_words = ["didn’t", "rowling", "like"]
print("Removing:", remove_words)
print("Unicode check on the apostrophe:", [hex(ord(c)) for c in remove_words[0]])

hp_filtered2 = [w for w in hp_filtered if w not in remove_words]
hp_counts2 = Counter(hp_filtered2)
top30_v2 = hp_counts2.most_common(30)

pd.DataFrame(top30_v2, columns=["Unigram", "Count"]).set_index("Unigram")

In [ ]:
# Why page, philosophers, stone and potter rank so high: a running page footer
footer_pos = hp_text.find("Page |")
print(repr(hp_text[footer_pos - 90:footer_pos + 70]))
print()
print("Occurrences of the footer phrase 'Page |':", hp_text.count("Page |"))

**What other words would you recommend removing?**

Three groups stand out.

**Page footer artifacts, the highest priority.** The document carries a repeating footer that reads
`Page | N Harry Potter and the Philosophers Stone - J.K. Rowling`, printed 347 times. That single
footer is what puts `page` (349), `philosophers` (347), `stone` (414) and `potter` (413) near the top
of the chart. Those counts describe the page layout of the file, not the language of the novel, so
`page`, `philosophers`, `potter` and most of `stone` should go. This is also why `rowling` had to be
removed in the first place, and it is the same root cause.

**Leftover contractions.** Because the curly apostrophe survived the punctuation strip, tokens such as
`he'd` (139), `don't` (112), `harry's` (111), `it's` (106) and `couldn't` are being treated as content
words. They are function words, and `harry's` is really just a possessive form of a name already in
the list. Either add U+2019 to the punctuation set or add these tokens to a custom stoplist.

**Low information verbs and fillers.** `said` (794) is by far the largest of these, and it tells you
the book is dialogue heavy but nothing about what the dialogue is about. `could`, `know`, `going`,
`think`, `look`, `looked` and `would` fall in the same category. Removing them would let character
names and setting words such as `hagrid`, `hermione`, `snape`, `dumbledore` and `professor` carry the
chart, which is a far more useful summary of the text.

### 8. Frequency plot of the top 30 unigrams (4 points)

In [ ]:
freq_df = pd.DataFrame(top30_v2, columns=["Unigram", "Count"])

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(freq_df["Unigram"], freq_df["Count"], color="#7F0909", edgecolor="#D3A625", linewidth=0.8)
ax.set_title("Top 30 Unigrams in Harry Potter and the Sorcerer's Stone", fontsize=15, pad=12)
ax.set_xlabel("Unigram", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.tick_params(axis="x", rotation=75)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 9. Re-create the frequency plot with cumulative counts (3 points)

In [ ]:
cum_df = freq_df.copy()
cum_df["Cumulative"] = cum_df["Count"].cumsum()

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(cum_df["Unigram"], cum_df["Cumulative"], color="#2E5A88", edgecolor="#D3A625", linewidth=0.8)
ax.set_title("Top 30 Unigrams, Cumulative Counts", fontsize=15, pad=12)
ax.set_xlabel("Unigram", fontsize=12)
ax.set_ylabel("Cumulative Count", fontsize=12)
ax.tick_params(axis="x", rotation=75)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("These 30 words account for", cum_df["Cumulative"].iloc[-1], "of",
      len(hp_filtered2), "filtered words",
      f"({cum_df['Cumulative'].iloc[-1] / len(hp_filtered2) * 100:.1f}%)")

### 10. Word cloud with Gryffindor colors (6 points)

Gryffindor scarlet and gold, supplied as RGB tuples: background `(116, 0, 1)` and text `(211, 166, 37)`.
`min_font_size=7` and `collocations=False` are set as required. Turning collocations off stops
`WordCloud` from inventing its own two word phrases, so the cloud reflects the unigram counts that
were actually computed above.

In [ ]:
GRYFFINDOR_SCARLET = (116, 0, 1)     # RGB tuple, background
GRYFFINDOR_GOLD = (211, 166, 37)     # RGB tuple, text

def gryffindor_color(word, font_size, position, orientation, random_state=None, **kwargs):
    """Return the Gryffindor gold RGB tuple for every word."""
    return GRYFFINDOR_GOLD

wc = WordCloud(
    width=1400,
    height=700,
    min_font_size=7,
    background_color=GRYFFINDOR_SCARLET,
    color_func=gryffindor_color,
    collocations=False,
    prefer_horizontal=0.9,
    max_words=200,
    random_state=42,
)
wc.generate_from_frequencies(hp_counts2)

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Harry Potter Unigrams, Gryffindor Word Cloud", fontsize=15, pad=12)
plt.tight_layout()
plt.show()

wc.to_file("harry_potter_wordcloud.png")
print("Saved harry_potter_wordcloud.png")

**According to the e-book, why are word clouds discouraged?**

A word cloud never gives you the number. The only quantity it encodes is the size of the text, and
size is one of the weakest visual channels people have for judging magnitude. Reading it, you can
tell that `harry` is the biggest word, but you cannot tell whether it appears twice as often as
`said` or four times as often, and you certainly cannot recover the value 1,507. The frequency plot
in item 8 gives you that immediately, because a bar encodes value as length along a common baseline.

The distortion compounds. A long word such as `philosophers` occupies far more area than a short word
such as `page` at the same font size, so the eye reads it as more important when the counts are
almost identical, 347 against 349. Position and rotation are decided by a layout algorithm that is
filling empty space, so a word near the center or set vertically carries no extra meaning even though
it looks like it does. There is no axis, no scale and no sort order to anchor a reader.

The cloud produced above proves the point on its own. `stone` and `philosophers` are drawn nearly
as large as `harry`, yet `harry` occurs 1,507 times against 414 and 347. `page`, a pure page footer
artifact, is rendered as one of the biggest words on the canvas. Nothing in the image warns you about
any of that, while the bar chart puts `harry` at almost twice the height of every other bar and makes
the problem impossible to miss.

The practical objection follows from all of that. A word cloud is decoration that looks like analysis.
It reads well on a poster or a title slide, but in a business setting where someone has to act on the
number, the frequency plot is the honest chart. Pretty versus practical, and practical wins when a
decision depends on it.

### 11. Create a list of bigrams and display the first ten (3 points)

In [ ]:
hp_bigrams = list(zip(hp_filtered2, hp_filtered2[1:]))

print("Total bigrams:", len(hp_bigrams))
hp_bigrams[:10]

### 12. Display the top 30 bigrams with their counts (3 points)

In [ ]:
bigram_counts = Counter(hp_bigrams)
top30_bigrams = bigram_counts.most_common(30)

pd.DataFrame(
    [(" ".join(b), n) for b, n in top30_bigrams],
    columns=["Bigram", "Count"]
).set_index("Bigram")

**A couple of bigrams that I would remove**

`potter philosophers` (347) and `philosophers stone` (347) are the two obvious ones. Neither is a
phrase anybody wrote in the novel. Both are fragments of the repeating page footer, and their counts
match the footer count exactly. `page harry` (347) is the same artifact, formed where one footer runs
into the start of the next block of text, and `harry potter` (359) is inflated by the same source,
since only about a dozen of those 359 are real in-text uses of the full name.

I would also drop `you” said` (16), which is not a word pair at all. It is the residue of the curly
right double quotation mark surviving the punctuation strip, so the token `you"` got glued to the
dialogue tag that follows it. After clearing those out, the list that remains is genuinely useful:
`uncle vernon` (95), `professor mcgonagall` (84), `aunt petunia` (52), `said hagrid` (89),
`common room` (20) and `crabbe goyle` (16) are real, meaningful phrases from the book.

---
# Part 2: Illuminati Wikipedia page

### 13. Parse the HTML and use `enumerate()` to display all links with index values (3 points)

In [ ]:
WIKI_URL = "https://en.wikipedia.org/wiki/Illuminati_(comics)"
HEADERS = {"User-Agent": "Mozilla/5.0 (coursework; data science assignment)"}

response = requests.get(WIKI_URL, headers=HEADERS, timeout=60)
print("Status code:", response.status_code)
print("Characters of HTML:", len(response.text))

soup = BeautifulSoup(response.text, "lxml")
all_links = soup.find_all("a")
print("Anchor tags found:", len(all_links))

In [ ]:
# enumerate() pairs each link with its index value
for index, link in enumerate(all_links):
    print(index, link)

### 14. All link elements with a `title` attribute of "Iron Man" (3 points)

In [ ]:
iron_man_title = soup.find_all("a", title="Iron Man")

print("Matches:", len(iron_man_title))
for link in iron_man_title:
    print(link)

### 15. All link elements with an exact text match of "Iron Man" (3 points)

In [ ]:
iron_man_text = soup.find_all("a", string="Iron Man")

print("Matches:", len(iron_man_text))
for link in iron_man_text:
    print(link)

### 16. All link elements containing "Strange" anywhere in the text (3 points)

In [ ]:
strange_anywhere = soup.find_all("a", string=re.compile("Strange"))

print("Matches:", len(strange_anywhere))
for link in strange_anywhere:
    print(repr(link.get_text()), "->", link.get("href"))

### 17. All link elements containing "Strange" at the end of the text (3 points)

The `$` anchor in the regular expression pins the match to the end of the string.

In [ ]:
strange_end = soup.find_all("a", string=re.compile("Strange$"))

print("Matches:", len(strange_end))
for link in strange_end:
    print(repr(link.get_text()), "->", link.get("href"))

print()
print("Dropped by the $ anchor:",
      [l.get_text() for l in strange_anywhere if l not in strange_end])

### 18. XPath: all link elements inside the Original Team table, and their hrefs (4 points)

`Original team` is the sub heading immediately under `Roster`. The XPath selects that heading by its
id, steps to the first `table` element that follows it in document order, and collects every anchor
inside it.

In [ ]:
tree = lh.fromstring(response.content)

xpath_expr = "//h3[@id='Original_team']/following::table[1]//a"
team_links = tree.xpath(xpath_expr)

print("XPath used:", xpath_expr)
print("Link elements found:", len(team_links))
print()
for a in team_links:
    print(f"{a.text_content():20} -> {a.get('href')}")

In [ ]:
# Build the full list of href attributes, made absolute so they can be requested directly
team_hrefs = [urljoin(WIKI_URL, a.get("href")) for a in team_links]

print(team_hrefs)

### 19. Combine the text from all those link pages and display the first 150 characters (3 points)

In [ ]:
page_texts = []

for url in team_hrefs:
    page = requests.get(url, headers=HEADERS, timeout=60)
    page_soup = BeautifulSoup(page.text, "lxml")
    body = page_soup.find("div", {"id": "mw-content-text"})
    body_text = body.get_text(" ", strip=True) if body else page_soup.get_text(" ", strip=True)
    page_texts.append(body_text)
    print(f"{url.split('/')[-1]:25} {len(body_text):>8,} characters")

wiki_text = " ".join(page_texts)
print()
print("Pages collected:", len(page_texts))
print("Combined characters:", f"{len(wiki_text):,}")
print()
print(wiki_text[:150])

### 20. Clean the text: punctuation, numbers, lowercase, split, stopwords, short words (4 points)

In [ ]:
wiki_no_punct = "".join(ch for ch in wiki_text if ch not in string.punctuation)
wiki_no_digits = "".join(ch for ch in wiki_no_punct if not ch.isdigit())
wiki_words = wiki_no_digits.lower().split()
wiki_filtered = [w for w in wiki_words if w not in stop_set and len(w) > 3]

print("Words before filtering:", f"{len(wiki_words):,}")
print("Words after filtering: ", f"{len(wiki_filtered):,}")
print("Unique words remaining:", f"{len(set(wiki_filtered)):,}")
print()
print(wiki_filtered[:25])

### 21. Display the top 25 unigrams with their counts (3 points)

In [ ]:
wiki_counts = Counter(wiki_filtered)
wiki_top25 = wiki_counts.most_common(25)

pd.DataFrame(wiki_top25, columns=["Unigram", "Count"]).set_index("Unigram")

### 22. Frequency plot of the top 25 unigrams (4 points)

In [ ]:
wiki_freq_df = pd.DataFrame(wiki_top25, columns=["Unigram", "Count"])

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(wiki_freq_df["Unigram"], wiki_freq_df["Count"], color="#0B3D91", edgecolor="#C8102E", linewidth=0.8)
ax.set_title("Top 25 Unigrams Across the Illuminati Original Team Pages", fontsize=15, pad=12)
ax.set_xlabel("Unigram", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.tick_params(axis="x", rotation=70)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

The top of this list is dominated by the five founding members and their affiliated teams, which is
what you would expect given where the links came from. Two entries are site furniture rather than
content: `retrieved` and `archived` come from citation templates in the reference lists, and `edit`
comes from the section edit links. Those three would be the first candidates for a custom stoplist.

### 23. Display the bottom 40 unigrams with their counts (3 points)

In [ ]:
wiki_bottom40 = wiki_counts.most_common()[-40:]

pd.DataFrame(wiki_bottom40, columns=["Unigram", "Count"]).set_index("Unigram")

In [ ]:
hapax = [w for w, n in wiki_counts.items() if n == 1]
print("Words appearing exactly once:", f"{len(hapax):,}",
      f"({len(hapax) / len(wiki_counts) * 100:.1f}% of the vocabulary)")

The bottom of the distribution is a long tail of words used exactly once. This is the standard shape
of natural language described by Zipf's law, where a small number of words do most of the work and
the majority of the vocabulary appears a single time. A few entries here are also cleanup artifacts,
such as the mangled URL token produced when punctuation was stripped out of a citation link.

### 24. Create a list of bigrams and display the top 20 with counts (3 points)

In [ ]:
wiki_bigrams = list(zip(wiki_filtered, wiki_filtered[1:]))
wiki_bigram_counts = Counter(wiki_bigrams)
wiki_top20_bigrams = wiki_bigram_counts.most_common(20)

print("Total bigrams:", f"{len(wiki_bigrams):,}")
pd.DataFrame(
    [(" ".join(b), n) for b, n in wiki_top20_bigrams],
    columns=["Bigram", "Count"]
).set_index("Bigram")

### 25. Frequency plot of the top 20 bigrams (4 points)

In [ ]:
wiki_bg_df = pd.DataFrame(
    [(" ".join(b), n) for b, n in wiki_top20_bigrams],
    columns=["Bigram", "Count"]
)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(wiki_bg_df["Bigram"][::-1], wiki_bg_df["Count"][::-1], color="#C8102E", edgecolor="#0B3D91", linewidth=0.8)
ax.set_title("Top 20 Bigrams Across the Illuminati Original Team Pages", fontsize=15, pad=12)
ax.set_xlabel("Count", fontsize=12)
ax.set_ylabel("Bigram", fontsize=12)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

Bigrams recover the phrases that single words break apart. `fantastic four`, `black bolt`,
`doctor strange`, `mister fantastic`, `captain america` and `marvel cinematic universe` are all
proper names that were invisible as unigrams, where `black` and `four` looked like generic filler.
The one to discard is `archived original`, which is boilerplate from the citation templates.

---
# Part 3: Disney API

### 26. Print out the JSON data (3 points)

In [ ]:
API_URL = "https://api.disneyapi.dev/character"

api_response = requests.get(API_URL, timeout=60)
print("Status code:", api_response.status_code)

disney_json = api_response.json()

print(json.dumps(disney_json, indent=2))

### 27. Display the keys in this Python dictionary (3 points)

In [ ]:
print("Type:", type(disney_json))
print("Keys:", list(disney_json.keys()))
print()
print("The 'info' key holds the pagination metadata:")
print(disney_json["info"])
print()
print("The 'data' key holds the character records. Records on this page:", len(disney_json["data"]))

### 28. Parse the data key into a data frame and display all records (3 points)

In [ ]:
disney_df = pd.DataFrame(disney_json["data"])

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 250):
    display(disney_df)

In [ ]:
print("Shape (rows, columns):", disney_df.shape)
print()
print("Number of variables in this data frame:", disney_df.shape[1])
print()
print("The variables are:")
for i, col in enumerate(disney_df.columns, start=1):
    print(f"  {i:2}. {col}")

**How many variables are in this data frame?**

There are **11 variables**: `_id`, `films`, `shortFilms`, `tvShows`, `videoGames`, `parkAttractions`,
`allies`, `enemies`, `name`, `imageUrl` and `url`. The default page returns **50 records**, and the
`info` key reports 197 total pages, so this is only the first slice of the full character catalog.
Seven of the eleven variables hold Python lists rather than single values, which matters in the
next step.

### 29. Save the data to an Excel file (6 points)

Requirements: no index values, sheet name `API Practice`, and the panes frozen so that the first row
and the first column stay locked while scrolling. `freeze_panes=(1, 1)` locks everything above row 1
and to the left of column 1, which is the header row and the `_id` column.

Excel cells cannot hold Python list objects, so the seven list columns are flattened into comma
separated strings first. Everything is done in code, with no manual editing of the workbook.

In [ ]:
excel_df = disney_df.copy()

list_columns = [c for c in excel_df.columns if excel_df[c].apply(lambda v: isinstance(v, list)).any()]
print("List valued columns flattened for Excel:", list_columns)

for col in list_columns:
    excel_df[col] = excel_df[col].apply(lambda v: ", ".join(map(str, v)) if isinstance(v, list) else v)

EXCEL_FILENAME = "disney_characters.xlsx"

excel_df.to_excel(
    EXCEL_FILENAME,
    sheet_name="API Practice",
    index=False,
    freeze_panes=(1, 1),
)

print("Saved", EXCEL_FILENAME)

In [ ]:
# Verify the file was written correctly
from openpyxl import load_workbook

wb = load_workbook(EXCEL_FILENAME)
ws = wb["API Practice"]

print("Sheet names:", wb.sheetnames)
print("Freeze panes setting:", ws.freeze_panes)
print("Dimensions:", ws.dimensions)
print("Header row:", [c.value for c in ws[1]])
print()
check = pd.read_excel(EXCEL_FILENAME, sheet_name="API Practice")
print("Rows read back:", len(check), " Columns read back:", check.shape[1])
print("Index column present?", "Unnamed: 0" in check.columns)
check.head()

---
## References

[1] Kaggle, "Harry Potter and the Sorcerer's Stone text." [Online]. Available: https://www.kaggle.com/

[2] Wikipedia, "Illuminati (comics)." [Online]. Available: https://en.wikipedia.org/wiki/Illuminati_(comics)

[3] Disney API, "Character endpoint." [Online]. Available: https://api.disneyapi.dev/character

[4] S. Bird, E. Klein, and E. Loper, *Natural Language Processing with Python*. Sebastopol, CA, USA:
O'Reilly Media, 2009. [Online]. Available: https://www.nltk.org/book/

[5] L. Richardson, "Beautiful Soup Documentation." [Online]. Available:
https://www.crummy.com/software/BeautifulSoup/bs4/doc/

[6] lxml, "XPath and XSLT with lxml." [Online]. Available: https://lxml.de/xpathxslt.html

[7] A. Mueller, "wordcloud: A little word cloud generator in Python." [Online]. Available:
https://amueller.github.io/word_cloud/

[8] The pandas development team, "pandas.DataFrame.to_excel." [Online]. Available:
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_excel.html